# Charge Density Visualization

Compare ground-truth HR, LR conditioning, and ResNet-predicted densities side by side using isosurface slices.


In [ ]:
import numpy as np
import torch
import yaml
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from pathlib import Path
from types import SimpleNamespace
from hydra.utils import instantiate

from electrai.lightning import LightningGenerator
from electrai.model.loss.charge import NormMAE


In [ ]:
# --- Configuration ---
CONFIG_PATH = "../src/electrai/configs/MP/config_resnet.yaml"
CKPT_PATH = "../checkpoints/last.ckpt"
SAMPLE_IDX = 0        # Which validation sample to visualize


In [ ]:
# Load config and data
with Path(CONFIG_PATH).open() as f:
    cfg = SimpleNamespace(**yaml.safe_load(f))

datamodule = instantiate(cfg.data)
datamodule.setup(stage="fit")
val_loader = datamodule.val_dataloader()
print(f"Validation set size: {len(datamodule.val_set)}")


In [ ]:
# Load model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
lit_model = LightningGenerator.load_from_checkpoint(CKPT_PATH, cfg=cfg)
lit_model = lit_model.to(device)
lit_model.eval()
print(f"Model loaded on {device}")


In [ ]:
# Get a specific validation sample and generate prediction
for i, batch in enumerate(val_loader):
    if i == SAMPLE_IDX:
        break

cond = batch["data"].to(device)     # LR conditioning
target = batch["label"].to(device)  # HR ground truth
index = batch["index"]

with torch.no_grad():
    pred = lit_model(cond)

loss_fn = NormMAE()
nmae = loss_fn(pred, target)
print(f"Sample index: {index}, NMAE: {nmae.item():.6f}")

# Move to numpy (squeeze batch and channel dims)
cond_np = cond[0, 0].cpu().numpy()
target_np = target[0, 0].cpu().numpy()
pred_np = pred[0, 0].cpu().numpy()
print(f"Grid shape: {target_np.shape}")


In [ ]:
def plot_density_slices(density, title, slice_axis=0):
    """Plot 3 orthogonal slices through the center of a 3D density."""
    nz, ny, nx = density.shape
    mid = [nz // 2, ny // 2, nx // 2]

    fig = make_subplots(rows=1, cols=3,
                        subplot_titles=[f"Z={mid[0]}", f"Y={mid[1]}", f"X={mid[2]}"])

    slices = [
        density[mid[0], :, :],  # XY plane
        density[:, mid[1], :],  # XZ plane
        density[:, :, mid[2]],  # YZ plane
    ]

    vmin = density.min()
    vmax = np.percentile(density, 99)  # clip top 1% for better contrast

    for col, s in enumerate(slices, 1):
        fig.add_trace(
            go.Heatmap(z=s, zmin=vmin, zmax=vmax, colorscale="Viridis",
                       showscale=(col == 3)),
            row=1, col=col,
        )

    fig.update_layout(title_text=title, height=350, width=1000)
    fig.show()

In [ ]:
plot_density_slices(cond_np, "LR Conditioning (Input)")

In [ ]:
plot_density_slices(target_np, "HR Ground Truth")

In [ ]:
plot_density_slices(pred_np, f"ResNet Prediction (NMAE={nmae.item():.4f})")


In [ ]:
# Difference map: prediction - ground truth
diff = pred_np - target_np
plot_density_slices(np.abs(diff), "Absolute Error |Pred - Target|")

In [ ]:
# 3D isosurface of the ground truth density
def plot_isosurface(density, title, n_surfaces=3):
    """Interactive 3D isosurface visualization."""
    nz, ny, nx = density.shape
    Z, Y, X = np.mgrid[0:nz, 0:ny, 0:nx]

    fig = go.Figure(data=go.Isosurface(
        x=X.flatten(), y=Y.flatten(), z=Z.flatten(),
        value=density.flatten(),
        isomin=float(np.percentile(density, 80)),
        isomax=float(np.percentile(density, 99)),
        surface_count=n_surfaces,
        opacity=0.4,
        caps=dict(x_show=False, y_show=False, z_show=False),
        colorscale="Viridis",
    ))
    fig.update_layout(title=title, height=600, width=700)
    fig.show()

plot_isosurface(target_np, "HR Ground Truth (3D Isosurface)")

In [ ]:
plot_isosurface(pred_np, f"ResNet Prediction (3D Isosurface, NMAE={nmae.item():.4f})")


In [ ]:
# Summary statistics
print(f"{'':20s} {'Cond (LR)':>12s} {'Target (HR)':>12s} {'Prediction':>12s}")
print("-" * 60)
for name, arr in [("Min", [cond_np.min(), target_np.min(), pred_np.min()]),
                   ("Max", [cond_np.max(), target_np.max(), pred_np.max()]),
                   ("Mean", [cond_np.mean(), target_np.mean(), pred_np.mean()]),
                   ("Std", [cond_np.std(), target_np.std(), pred_np.std()]),
                   ("Sum (electrons)", [cond_np.sum(), target_np.sum(), pred_np.sum()])]:
    print(f"{name:20s} {arr[0]:12.4f} {arr[1]:12.4f} {arr[2]:12.4f}")